In [ ]:
sim = 'sim1005'  # current steps for firing curve control
sim = 'sim1006'  # current steps for firing curve 20% KCNQ

In [ ]:
############################################## SETTINGS ##############################################  
# settings
cell_type='dspn'
model = 3

import os, sys
# compute absolute path of main folder
cwd = os.getcwd()

if os.path.exists(os.path.join(cwd, 'settings.py')):
    main_dir = cwd
else:
    main_dir = os.path.abspath(os.path.join(cwd, '..'))

# set CWD or add to path
os.chdir(main_dir)
sys.path.insert(0, main_dir)

# then import/run settings
%run -i settings.py

from analysis_functions import *

# Get the current working directory
current_wd = os.getcwd()
# 'sim' + str(sim)

downsample = True

home = os.path.expanduser('~')

# Set working directory
external = False
external_name = 'MacOS10'
target = f'model{model}'

if not external:
    if downsample:
        wd = home + '/Documents/Repositories/msNEURON_Belal2026/downsample/' + cell_type + '/' + target + '/physiological/simulations'
    else:
        wd = home + '/Documents/Repositories/msNEURON_Belal2026/' + cell_type + '/' + target + '/physiological/simulations'
else:
    if downsample:
        wd = '/Volumes/' + external_name + '/Repositories/msNEURON_Belal2026/downsample/' + cell_type + '/' + target + '/physiological/simulations'
    else:
        wd = '/Volumes/' + external_name + '/Repositories/msNEURON_Belal2026/' + cell_type + '/' + target + '/physiological/simulations'


# create path to directory to save analysis
base_path = os.path.join(home, 'Documents', 'Repositories', 'msNEURON_Belal2026', 'analysis')
sim_image_path = os.path.join(base_path, cell_type, sim)
os.makedirs(sim_image_path, exist_ok=True)

save = False

In [ ]:
# calculate surface area of a the neuron
# note NEURON’s default spatial units are micrometres (µm) so area returns µm²

cell, spines, dend_tree = cell_build(cell_type=cell_type, specs=specs, addSpines=True, spine_per_length=spine_per_length, frequency=frequency, d_lambda=d_lambda, verbose=False, dend2remove=None)
area, total_length = surface_area(cell=cell, spines=spines)

# draw cell

cell_coordinates = []

for sec in cell.somalist:
    h('access ' + sec.name())
    x, y, z, diam = interpolate_3d(sec, 0.5)  # Use 0.5 to refer to the center of the section
    cell_coordinates.append([sec.name(), 0.5, x, y, z, h.distance(0.5, sec=sec), diam])

# Setup for dendritic sections
for sec in cell.dendlist:
    for seg in sec:
        x, y, z, diam = interpolate_3d(sec, seg.x)
        cell_coordinates.append([sec.name(), seg.x, x, y, z, h.distance(seg.x, sec=sec), diam])
    
cell_coordinates = np.array(cell_coordinates, dtype=object)

    
fig_morphology = morphology_plot(cell_coordinates, dend_tree, lwd=0.8, s=2, color='grey', height=600, width=600, plane='xy', mirror=False)

fig_morphology.show(config={"toImageButtonOptions": {"format": "svg"}})
if save:
    fig_morphology.write_image(f"{sim_image_path}/morphology.svg", format='svg', scale=1)

    

In [ ]:
# # Test Kir current at different voltages under ideal voltage-clamp conditions created by finitialize(v)
# # gives the steady state values (independent of other conductances)

# # make sure a section is accessed
# soma = list(cell.somalist)[0]

# for v in range(-130, 0, 5):
#     h.finitialize(v)
#     h.fcurrent()
#     print(v, soma(0.5).ik_kir)




In [ ]:
col = '#595959'
lw = 1

def plot_xy(x, y, ylabel='', xlabel='',
            width=400, height=400, lw=lw, fig_title='',
            dash=None, xrange=None, yrange=None, yaxis_side='left',
            col='#595959'):
    fig = go.Figure()

    fig.add_trace(go.Scatter(
        x=x, y=y,
        mode='lines',
        line=dict(color=col, width=lw, dash=dash),
        name=ylabel
    ))

    xmin, xmax = x.min(), x.max()
    tickx = np.arange(np.floor(xmin / 10) * 10, np.ceil(xmax / 10) * 10 + 1, 10)
    ticky = np.linspace(y.min(), y.max(), 6)
    ypad = (ticky[-1] - ticky[0]) * 0.02 if (ticky[-1] - ticky[0]) != 0 else 1e-12

    fig.update_layout(
        autosize=False,
        width=width, height=height,
        margin=dict(l=60, r=60, t=40, b=60),
        paper_bgcolor='rgba(0,0,0,0)',
        plot_bgcolor='rgba(0,0,0,0)',
        title=dict(text=fig_title, x=0.5, xanchor='center',
                   font=dict(color=col, size=12)),
        xaxis=dict(title=xlabel, showgrid=False, zeroline=False,
                   linecolor=col, ticks='outside', automargin=True,
                   tickmode='array', tickvals=tickx,
                   ticktext=[str(int(v)) for v in tickx],
                   range=xrange if xrange is not None else [xmin, xmax]),
        yaxis=dict(title=ylabel, showgrid=False, zeroline=False,
                   side=yaxis_side, linecolor=col, ticks='outside',
                   tickformat='.0e', automargin=True,
                   tickvals=ticky, tickmode='array',
                   range=yrange if yrange is not None else [ticky[0], ticky[-1] + ypad]),
        font=dict(family='Myriad Pro', size=10, color=col),
        legend=dict(x=1.2, y=0.5, xanchor='left', yanchor='middle', borderwidth=0)
    )

    fig.update_xaxes(showline=True, linecolor=col, showgrid=False, zeroline=False, ticks='outside')
    fig.update_yaxes(showline=True, linecolor=col, showgrid=False, zeroline=False, ticks='outside')
    fig.update_traces(line_simplify=False, cliponaxis=False)
    fig.update_layout(legend_traceorder='normal', legend_tracegroupgap=0)
    fig.update_layout(annotationdefaults=dict(visible=True))

    return fig


soma = list(cell.somalist)[0]
ek = soma(0.5).ek
gbar_all = {
    'kir': soma(0.5).gbar_kir,
    'kcnq': soma(0.5).gbar_kcnq,
    'kaf': soma(0.5).gbar_kaf,
    'kas': soma(0.5).gbar_kas,
}

v_range = np.arange(-200, 200, 5)
results = {}

for name, gbar in gbar_all.items():
    i_list, g_list, m_list, h_list = [], [], [], []

    for v in v_range:
        try:
            h.finitialize(v)
            h.fcurrent()
            ik = getattr(soma(0.5), f'ik_{name}', np.nan)
        except Exception:
            ik = np.nan

        den = v - ek
        g = ik / den if abs(den) > 1e-9 else np.nan
        i_list.append(ik)
        g_list.append(g)

        m_val = getattr(soma(0.5), f'm_{name}', np.nan)
        h_val = getattr(soma(0.5), f'h_{name}', np.nan)
        m_list.append(m_val)
        h_list.append(h_val)

    results[name] = {
        'i': np.array(i_list),
        'g': np.array(g_list),
        'm': np.array(m_list),
        'h': np.array(h_list)
    }

for name, data in results.items():

    name_map = {
        'kir': 'K<sub>ir</sub>',
        'kcnq': 'K<sub>v7</sub>',
        'kaf': 'K<sub>v4</sub>',
        'kas': 'K<sub>v1</sub>',
        'bk': 'BK',
        'sk': 'SK'
    }
    title_name = name_map.get(name.lower(), name)

    has_h = not np.isnan(data['h']).all()

    figs = [
        plot_xy(v_range, data['i'], ylabel=f'I (mA/cm²)', xlabel='holding potential (mV)', col=col, lw=lw),
        plot_xy(v_range, data['g'], ylabel=f'g (S/cm²)', xlabel='holding potential (mV)', col=col, lw=lw),
        plot_xy(v_range, data['m'], ylabel=f'm∞', xlabel='holding potential (mV)', col=col, lw=lw)
    ]

    if has_h:
        figs.append(plot_xy(v_range, data['h'], ylabel=f'h∞', xlabel='holding potential (mV)', dash='dot', col=col, lw=lw))

    for f in figs:
        f.update_layout(width=400, height=400)

    from plotly.subplots import make_subplots
    from plotly import graph_objects as go

    fig_mechs1 = make_subplots(rows=1, cols=len(figs), subplot_titles=[f.layout.title.text for f in figs])

    for i, f in enumerate(figs, start=1):
        fig_mechs1.add_traces(f.data, rows=1, cols=i)
        fig_mechs1.layout[f'xaxis{i}'].title.text = f.layout.xaxis.title.text
        fig_mechs1.layout[f'yaxis{i}'].title.text = f.layout.yaxis.title.text

        fig_mechs1.add_hline(
            y=0,
            row=1,
            col=i,
            line=dict(color=col, width=lw, dash='dot')
        )

        fig_mechs1.add_vline(
            x=0,
            row=1,
            col=i,
            line=dict(color=col, width=lw, dash='dot')
        )
    
    fig_mechs1.update_layout(
        width=400 * len(figs),
        height=400,
        showlegend=False,
        font=dict(family='Myriad Pro', size=10, color=col),
        paper_bgcolor='rgba(0,0,0,0)',
        plot_bgcolor='rgba(0,0,0,0)',
        title=dict(
            text=title_name,
            x=0.5,
            xanchor='center',
            font=dict(size=12, color=col)
        )
    )

    for ax in fig_mechs1.layout:
        if ax.startswith('xaxis') or ax.startswith('yaxis'):
            fig_mechs1.layout[ax].update(
                showline=True,
                linecolor=col,
                ticks='outside',
                showgrid=False,
                zeroline=False
            )

    fig_mechs1.show(config={"toImageButtonOptions": {"format": "svg"}})
    if save:
        fig_mechs1.write_image(f"{sim_image_path}/fig_mechs1.svg", format='svg', scale=1)


In [ ]:
v_range = np.arange(-90, -50, 1)
results = {}

for name, gbar in gbar_all.items():
    i_list, g_list, m_list, h_list = [], [], [], []

    for v in v_range:
        try:
            h.finitialize(v)
            h.fcurrent()
            ik = getattr(soma(0.5), f'ik_{name}', np.nan)
        except Exception:
            ik = np.nan

        den = v - ek
        g = ik / den if abs(den) > 1e-9 else np.nan
        i_list.append(ik)
        g_list.append(g)

        # safely extract gating variables
        m_val = getattr(soma(0.5), f'm_{name}', np.nan)
        h_val = getattr(soma(0.5), f'h_{name}', np.nan)
        m_list.append(m_val)
        h_list.append(h_val)

    results[name] = {
        'i': np.array(i_list),
        'g': np.array(g_list),
        'm': np.array(m_list),
        'h': np.array(h_list)
    }
    
for name, data in results.items():

    name_map = {
        'kir': 'K<sub>ir</sub>',
        'kcnq': 'K<sub>v7</sub>',
        'kaf': 'K<sub>v4</sub>',
        'kas': 'K<sub>v1</sub>',
        'bk': 'BK',
        'sk': 'SK'
    }
    title_name = name_map.get(name.lower(), name)

    has_h = not np.isnan(data['h']).all()

    figs = [
        plot_xy(v_range, data['i'], ylabel=f'I (mA/cm²)', xlabel='holding potential (mV)', col=col, lw=lw),
        plot_xy(v_range, data['g'], ylabel=f'g (S/cm²)', xlabel='holding potential (mV)', col=col, lw=lw),
        plot_xy(v_range, data['m'], ylabel=f'm∞', xlabel='holding potential (mV)', col=col, lw=lw)
    ]

    if has_h:
        figs.append(plot_xy(v_range, data['h'], ylabel=f'h∞', xlabel='holding potential (mV)', dash='dot', col=col, lw=lw))

    for f in figs:
        f.update_layout(width=400, height=400)

    from plotly.subplots import make_subplots
    from plotly import graph_objects as go

    fig_mechs2 = make_subplots(rows=1, cols=len(figs), subplot_titles=[f.layout.title.text for f in figs])

    for i, f in enumerate(figs, start=1):
        fig_mechs2.add_traces(f.data, rows=1, cols=i)
        fig_mechs2.layout[f'xaxis{i}'].title.text = f.layout.xaxis.title.text
        fig_mechs2.layout[f'yaxis{i}'].title.text = f.layout.yaxis.title.text

        fig_mechs2.add_hline(
            y=0,
            row=1,
            col=i,
            line=dict(color=col, width=lw, dash='dot')
        )
    
    fig_mechs2.update_layout(
        width=400 * len(figs),
        height=400,
        showlegend=False,
        font=dict(family='Myriad Pro', size=10, color=col),
        paper_bgcolor='rgba(0,0,0,0)',
        plot_bgcolor='rgba(0,0,0,0)',
        title=dict(
            text=title_name,
            x=0.5,
            xanchor='center',
            font=dict(size=12, color=col)
        )
    )

    for ax in fig_mechs2.layout:
        if ax.startswith('xaxis') or ax.startswith('yaxis'):
            fig_mechs2.layout[ax].update(
                showline=True,
                linecolor=col,
                ticks='outside',
                showgrid=False,
                zeroline=False
            )

    fig_mechs2.show(config={"toImageButtonOptions": {"format": "svg"}})

    if save:
        fig_mechs2.write_image(f"{sim_image_path}/fig_mechs2.svg", format='svg', scale=1)

    

In [ ]:
# use load_data_dicts to load data
sims_dir = os.path.join(wd, sim)
data_dict = load_data_dicts(wd, sim, cell_type, verbose=True)

In [ ]:
# check memory usage
report_memory(verbose=True)

In [ ]:
# basic experimental metadata
ds = data_dict['metadata']['ds']
dt = data_dict['metadata']['dt']
dt = ds * dt

sim_time = data_dict['metadata']['sim_time']
step_end = sim_time - 200
step_duration = 1000
step_start = step_end - step_duration
step_current = -10

holding_current_range = list(range(-100, 500, 20))


mechs2extract = ['kaf', 'kas', 'kir', 'kcnq', 'sk', 'bk']

print(step_start, step_duration, step_end)


In [ ]:
# generate plots
x = np.arange(0,len(data_dict['vsoma'][0])*dt, dt)
_range_subset = [i for i in range(0, len(holding_current_range), 2)]


plot9_MLP(x=x, ydict=data_dict['vsoma'], _range=holding_current_range, _range_subset=_range_subset, yabline=[-85, -60], ds=1,
                xaxis_range=[step_start-100, sim_time], yaxis_range=[-95, -45], sim_image_path=sim_image_path, save_name='fig_summary1.svg', save=save)



In [ ]:
# select 4 evenly space; one at rest one at -70 and one at -60 mV
if cell_type == 'dspn':
    _range_subset=[6, 10, 12, 14]
elif cell_type == 'ispn':
    _range_subset=[6, 10, 12, 14]

plot9_MLP(x=x,ydict=data_dict['vsoma'], _range=holding_current_range, _range_subset=_range_subset, ds=1,
                    xaxis_range=[step_start-100, sim_time], yaxis_range=[-90,-60], yabline=[-85, -60],
                    sim_image_path=sim_image_path, save_name='fig_summary2.svg', save=save)



In [ ]:
for mech in mechs2extract:
    plot_mech_current_MLP(mech=mech, data_dict=data_dict, _range=holding_current_range, _range_subset=_range_subset, ds=1,
                      sim_image_path=sim_image_path, sim_time=sim_time, step_start=step_start, dt=dt, save=save)

In [ ]:
import matplotlib
print(matplotlib.get_backend())

In [ ]:
sample_window = 2.5  # ms
nsamples = int(sample_window / dt)

v_pre = []
v_post = []

for vsoma in data_dict['vsoma']:
    pre_segment = vsoma[int(step_start/dt) - nsamples : int(step_start/dt)]
    post_segment = vsoma[int(step_end/dt) - nsamples : int(step_end/dt)]
    v_pre.append(np.mean(pre_segment))
    v_post.append(np.mean(post_segment))

v_pre = np.array(v_pre)
v_post = np.array(v_post)

dV = v_post - v_pre
Vm = v_pre
I = np.array(holding_current_range)  # pA

IR_MOhm = (dV / step_current) * 1e3  # MΩ (mV/pA * 1e3)

df1 = pd.DataFrame({'I': I, 'Vm': Vm, 'dV': dV, 'IR': IR_MOhm})
df1

In [ ]:
import json

if cell_type == 'dspn':
    params='params_dMSN3.json'
elif cell_type == 'ispn':
    params='params_iMSN3.json'     

with open(params, 'r') as f:
    params_dict = json.load(f)

ek = float(params_dict['ek']['Value'])
vsoma = data_dict['vsoma']

g_pas = float(params_dict['g_pas_all']['Value'])
grest_dict = {}

grest_dict = {}
irest_dict = {}

for mech in mechs2extract:
    isoma = extract_mechs_dict2array(data_dict['i_mechs_dend'], mech=mech)
    gsoma = [i / (v - ek) for i, v in zip(isoma, vsoma)]
    gsoma = [np.nan_to_num(g, nan=0.0, posinf=0.0, neginf=0.0) for g in gsoma]

    i_pre, g_pre = [], []
    for i, g in zip(isoma, gsoma):
        seg = slice(int(step_start/dt) - nsamples, int(step_start/dt))
        i_pre.append(np.mean(i[seg]))
        g_pre.append(np.mean(g[seg]))

    irest_dict[mech] = i_pre
    grest_dict[mech] = g_pre


df2 = pd.DataFrame({'I': I, 'Vm': Vm})
for mech in mechs2extract:
    df2[f'grest_{mech}'] = grest_dict[mech]

df3 = pd.DataFrame({'I': I, 'Vm': Vm})
for mech in mechs2extract:
    df3[f'irest_{mech}'] = irest_dict[mech]

df3

In [ ]:
if sim == 'sim1005':
    open_circles=False
    dashed_lines=False
elif sim == 'sim1006':
    open_circles=True
    dashed_lines=True
    
plot_xy_MLP(df1['Vm'], df1['IR'], sim_image_path=sim_image_path, save=True, y_title='Conductance (S/cm²)', 
            xrange=[-100, -40], yrange=[0, 300], open_circles=open_circles, dashed_lines=dashed_lines, x_title='Membrane potential (mV)', title='IR vs Vm', 
            save_name='ir_vs_vm_summary.svg')



In [ ]:
y_cols = [f"grest_{m}" for m in mechs2extract]
plot_df(df2, x_col='Vm', y_cols=y_cols, sim_image_path=sim_image_path, save=True, y_title='Conductance (S/cm²)', marker_size=4,
        xrange =(-100, -40), yrange=(1e-10, 1e-3), open_circles=open_circles, dashed_lines=dashed_lines, x_title='Membrane potential (mV)', 
        title='Conductance vs Vm', log_y=True, save_name='conductance_vs_vm_summary.svg')

In [ ]:
df3

In [ ]:
y_cols = [f"irest_{m}" for m in mechs2extract]
plot_df(df3, x_col='Vm', y_cols=y_cols, sim_image_path=sim_image_path, save=True, y_title = 'i (µA/cm²)', marker_size=4,
    xrange=(-100, -40), yrange=(-0.001, 0.0025), open_circles=open_circles, dashed_lines=dashed_lines, x_title='membrane potential (mV)', title='current density vs Vm', 
    save_name='current_density_vs_vm_summary.svg')


In [ ]:
# constants
v_rest = -85
v_test = -60
g_pas = float(params_dict['g_pas_all']['Value'])

# whole cell area 
area_um2 = area 
area_cm2 = area_um2 * 1e-8 

# # soma area
# area_cm2 = sum(h.area(seg.x, sec=cell.soma) for seg in cell.soma) * 1e-8

mechs = ['kaf', 'kas', 'kir', 'kcnq', 'sk', 'bk']

def interp_g(vm_target):
    g_vals = {}
    for m in mechs:
        col = f"grest_{m}"
        if col in df2.columns:
            g_vals[m] = np.interp(vm_target, df2['Vm'], df2[col])
        else:
            g_vals[m] = np.nan
    g_vals['pas'] = g_pas
    return g_vals

def compute_summary(g_dict):
    total = np.nansum(list(g_dict.values()))              
    Rin = 1 / (total * area_cm2) / 1e6 if total > 0 else np.nan 
    return total, Rin

# at rest
g_rest = interp_g(v_rest)
total_g_rest, Rin_rest = compute_summary(g_rest)

print(f"\nResting membrane potential: {v_rest} mV")
for k, g in g_rest.items():
    pct = 100 * g / total_g_rest
    print(f"  {k:<6} {g:.4e} S/cm² ({pct:.2f}%)")
print(f"  total_g_rest = {total_g_rest:.4e} S/cm²")
print(f"  Rin_rest     = {Rin_rest:.1f} MΩ")

# at -60 mV
g_test = interp_g(v_test)
total_g_test, Rin_test = compute_summary(g_test)

print(f"\nAt test potential: {v_test} mV")
for k, g in g_test.items():
    pct = 100 * g / total_g_test
    print(f"  {k:<6} {g:.4e} S/cm² ({pct:.2f}%)")
print(f"  total_g_test = {total_g_test:.4e} S/cm²")
print(f"  Rin_test     = {Rin_test:.1f} MΩ\n")

In [ ]:
# assume cai = 5e-5 (mM) is NEURON resting intracellular Ca²⁺ level.
cai = 5e-5

gbar = {}
for k in ['kaf', 'kas', 'kir', 'kcnq', 'bk', 'sk', 'pas']:
    key = f"gbar_{k}_somatic" if k != 'pas' else 'g_pas_all'
    val = float(params_dict[key]['Value'])
    sf = scale_factors.get(k, 1) if 'scale_factors' in locals() and isinstance(scale_factors, dict) else 1
    gbar[k] = val * sf


def kaf_g(v, gbar):
    a_m = 1.5 / (1 + np.exp((v - 5) / -17))
    b_m = 0.6 / (1 + np.exp((v - 11) / 9))
    m_inf = a_m / (a_m + b_m)
    a_h = 0.105 / (1 + np.exp((v + 120) / 22))
    b_h = 0.065 / (1 + np.exp((v + 54) / -11))
    h_inf = a_h / (a_h + b_h)
    return gbar * (m_inf**2) * h_inf


def kas_g(v, gbar):
    a_m = 0.25 / (1 + np.exp((v - 50) / -20))
    b_m = 0.05 / (1 + np.exp((v + 90) / 35))
    m_inf = a_m / (a_m + b_m)
    a_h = 0.0025 / (1 + np.exp((v + 95) / 16))
    b_h = 0.002 / (1 + np.exp((v - 50) / -70))
    h_inf = 0.2 + (a_h / (a_h + b_h)) * 0.8
    return gbar * (m_inf**2) * h_inf


def kir_g(v, gbar):
    m_inf = 1 / (1 + np.exp((v + 102) / 13))
    return gbar * m_inf


def kcnq_g(v, gbar):
    m_inf = 1 / (1 + np.exp((-59.5 - v) / 10.3))
    return gbar * (m_inf**4)


def sk_g(v, gbar, cai=5e-5):
    v = np.asarray(v)
    a = (cai / 0.57e-3) ** 5.2
    o_inf = a / (1 + a)
    return gbar * np.ones_like(v) * o_inf


def bk_g(v, gbar, cai=5e-5, celsius=35):
    v = np.asarray(v)
    F = 96485
    R = 8.314
    k1 = 0.180e-3
    k4 = 0.011e-3
    z = 1e-3 * 2 * F / (R * (celsius + 273.15))
    a = 0.48 * cai / (cai + k1 * np.exp(-0.84 * z * v))
    b = 0.28 / (1 + cai / (k4 * np.exp(-z * v)))
    o_inf = a / (a + b)
    return gbar * o_inf


v_rest = -85
v_test = -60

Cm = 1e-6 * area_cm2 * 1e12
funcs = {'kaf': kaf_g, 'kas': kas_g, 'kir': kir_g, 'kcnq': kcnq_g, 'sk': sk_g, 'bk': bk_g}

g_rest = {k: f(v_rest, gbar[k]) for k, f in funcs.items()}
g_rest['pas'] = gbar['pas']
total_g_rest = sum(g_rest.values())
Rin_rest = 1 / (total_g_rest * area_cm2) / 1e6

g_60 = {k: f(v_test, gbar[k]) for k, f in funcs.items()}
g_60['pas'] = gbar['pas']
total_g_60 = sum(g_60.values())
Rin_60 = 1 / (total_g_60 * area_cm2) / 1e6

v_range = np.linspace(-90, -40, 200)
Rin_vals = []
for v in v_range:
    g_tot = sum([f(v, gbar[k]) for k, f in funcs.items()]) + gbar['pas']
    Rin_vals.append(1 / (g_tot * area_cm2) / 1e6)

V_th = -59.5
ΔV_rest = V_th - v_rest
lo, hi = sorted([v_rest, V_th])
mask = (v_range >= lo) & (v_range <= hi)
Rin_path = np.array(Rin_vals)[mask]
Rin_peak = float(np.max(Rin_path))
Rin_at_th = float(np.interp(V_th, v_range, Rin_vals))

xrange_common = [-100, -40]


# Input resistance vs Vm
plot_xy_MLP(x=pd.Series(v_range), y=pd.Series(Rin_vals), sim_image_path=sim_image_path, save=True,
    y_title='Input resistance (MΩ)', open_circles=open_circles, dashed_lines=dashed_lines, x_title='Membrane potential (mV)',
    title='IR vs Vm',save_name='ir_vs_vm_summary2.svg', markers=False)


In [ ]:
# Conductance vs Vm (semilog)
df2 = pd.DataFrame({'Vm': v_range})
for m in funcs.keys():
    df2[f"grest_{m}"] = funcs[m](v_range, gbar[m])

y_cols = [f"grest_{m}" for m in funcs.keys()]
plot_df(df2, x_col='Vm', y_cols=y_cols, sim_image_path=sim_image_path, save=True, y_title='Conductance (S/cm²)',
    xrange=(-100, -40), yrange=(1e-10, 1e-3), open_circles=open_circles, dashed_lines=dashed_lines, x_title='Membrane potential (mV)',
    title='Conductance vs Vm', log_y=True, save_name='conductance_vs_vm_summary2.svg', markers=False)



In [ ]:

# print(f"\nResting membrane potential: {v_rest} mV")
# for k, g in g_rest.items():
#     pct = 100 * g / total_g_rest
#     print(f"  {k:<6} {g:.4e} S/cm² ({pct:.2f}%)")
# print(f"  total_g_rest = {total_g_rest:.4e} S/cm²")
# print(f"  Rin_rest     = {Rin_rest:.1f} MΩ")

# g_test = {k: f(v_test, gbar[k]) for k, f in funcs.items()}
# g_test['pas'] = gbar['pas']
# total_g_test = sum(g_test.values())
# Rin_test = 1 / (total_g_test * 1e-4) / 1e6

# print(f"\nAt test potential: {v_test} mV")
# for k, g in g_test.items():
#     pct = 100 * g / total_g_test
#     print(f"  {k:<6} {g:.4e} S/cm² ({pct:.2f}%)")
# print(f"  total_g_test = {total_g_test:.4e} S/cm²")
# print(f"  Rin_test     = {Rin_test:.1f} MΩ\n")

If linear-scale plot looks exponential on a semilog axis (i.e. straight line on log scale), that means the conductance changes exponentially with voltage — usually indicating strong voltage dependence, which can correspond to either inward or outward rectification depending on direction.

In [ ]:
v_neg = -100
v_pos = -40

rectification = {}
for name, func in funcs.items():
    g_neg = func(v_neg, gbar[name])
    g_pos = func(v_pos, gbar[name])
    ratio = g_neg / g_pos if g_pos != 0 else np.nan
    if ratio > 1.2:
        rect_type = 'inward rectifier'
    elif ratio < 0.8:
        rect_type = 'outward rectifier'
    else:
        rect_type = 'linear (non-rectifying)'
    rectification[name] = (ratio, rect_type)

print('Rectification indices:')
for name, (ratio, rect_type) in rectification.items():
    print(f"{name:5s}: G(-100)/G(-40) = {ratio:.2f} : {rect_type}")

In [ ]:
# # create a html version
# if save:
#     from pathlib import Path

#     nb_name = "passive tune analysis Matplotlib.ipynb"
#     nb_path = Path(main_dir) / "analysis notebooks" / nb_name   # adjust folder if needed

#     output_dir = Path(main_dir) / "docs" / cell_type
#     output_dir.mkdir(parents=True, exist_ok=True)

#     if not nb_path.exists():
#         raise FileNotFoundError(f"Notebook not found: {nb_path}")

#     !jupyter nbconvert "{nb_path}" --to html --output-dir="{output_dir}"

